## Reading the CSV

In [127]:
import pandas as pd

df = pd.read_csv("cars.csv")
df.head()

,url,price,brand,series,model,year,mileage,transmission,fuel,body,...,engine,hp,drive,condition,heavy_damage,paint_changed,fuel_consumption,fuel_tank,trade_in,seller_type
0,https://www.arabam.com/ilan/galeriden-satilik-...,1.049.000 TL,Ford,Focus,1.5 TDCi Trend X,2018,180.000 km,Otomatik,Dizel,Sedan,...,1401 - 1600 cm3,101 - 125 HP,Önden Çekiş,İkinci El,NaN,"1 değişen, 2 boyalı",NaN,NaN,Takasa Uygun,Galeriden
1,https://www.arabam.com/ilan/galeriden-satilik-...,555.750 TL,Ford,Focus,1.6 TDCi Collection,2010,357.000 km,Düz,Dizel,Hatchback/5,...,1560 cc,90 hp,Önden Çekiş,İkinci El,Hayır,5 boyalı,"4,7 lt",55 lt,Takasa Uygun,Galeriden
2,https://www.arabam.com/ilan/galeriden-satilik-...,595.000 TL,Citroen,C-Elysée,1.6 HDi Attraction,2014,225.000 km,Düz,Dizel,Sedan,...,1560 cc,93 hp,Önden Çekiş,İkinci El,NaN,2 boyalı,"4,3 lt",48 lt,Takasa Uygun,Galeriden
3,https://www.arabam.com/ilan/galeriden-satilik-...,2.059.000 TL,Mercedes - Benz,C,C 180 BlueEFFICIENCY AMG,2014,75.000 km,Otomatik,Benzin,Coupe,...,1401 - 1600 cm3,151 - 175 HP,Arkadan İtiş,İkinci El,Belirtilmemiş,Tamamı orjinal,NaN,NaN,Takasa Uygun,Galeriden
4,https://www.arabam.com/ilan/galeriden-satilik-...,1.150.000 TL,Volvo,S60,1.6 D Advance,2014,193.000 km,Otomatik,Dizel,Sedan,...,1401 - 1600 cm3,101 - 125 HP,Önden Çekiş,İkinci El,NaN,1 değişen,NaN,NaN,Takasa Uygun,Galeriden


## Cleaning the Strings

### We want to clean the price and mileage data and convert them to floats

In [128]:
df["price"] = df["price"].str.replace(" TL", "").str.replace(".", "").astype(float)
price.head()

0    1049000.0
1     555750.0
2     595000.0
3    2059000.0
4    1150000.0
Name: price, dtype: float64

In [129]:
df["mileage"] = df["mileage"].str.replace(" km", "").str.replace(".", "").astype(float)
df["mileage"].head()

0    180000.0
1    357000.0
2    225000.0
3     75000.0
4    193000.0
Name: mileage, dtype: float64

In [130]:
df['heavy_damage'] = df['heavy_damage'].fillna('Belirtilmemiş')
df["heavy_damage"].describe()

count              1877
unique                3
top       Belirtilmemiş
freq               1156
Name: heavy_damage, dtype: object

## Concatenate Brand & Series

### I decided to concatenate these two since each brand's series are special to that brand.

In [138]:
df["brand_series"] = df["brand"] + "_" + df["series"]
df["brand_series"].describe()

count          1877
unique          120
top       Fiat_Egea
freq            118
Name: brand_series, dtype: object

## Encoding Brand & Series

### We want to encode these to labels for our model to understand since working with strings does not make sense.

In [132]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["brand_series_encoded"] = le.fit_transform(df["brand_series"])
df["heavy_damage_encoded"] = le.fit_transform(df["heavy_damage"])
print(df["brand_series_encoded"].head())
print(df["heavy_damage_encoded"].head())

0     43
1     43
2      7
3     67
4    113
Name: brand_series_encoded, dtype: int64
0    0
1    2
2    0
3    0
4    0
Name: heavy_damage_encoded, dtype: int64


## Creating the Train & Test Splits

In [133]:
X = df[['brand_series_encoded', 'year', 'mileage', "heavy_damage_encoded"]]
y = df["price"]

print(X.head())
print(y.head())

   brand_series_encoded  year   mileage  heavy_damage_encoded
0                    43  2018  180000.0                     0
1                    43  2010  357000.0                     2
2                     7  2014  225000.0                     0
3                    67  2014   75000.0                     0
4                   113  2014  193000.0                     0
0    1049000.0
1     555750.0
2     595000.0
3    2059000.0
4    1150000.0
Name: price, dtype: float64


In [134]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)

## Training the baseline decision tree

In [135]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error

dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)

y_pred = dt.predict(X_test)
print("Decision Tree MAE:", mean_absolute_error(y_test, y_pred))

Decision Tree MAE: 214754.49468085106


## Training the baseline XGBoost

In [136]:
from xgboost import XGBRegressor

xgb = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6)
xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)
print("XGBoost MAE:", mean_absolute_error(y_test, y_pred_xgb))

XGBoost MAE: 169231.68828956116


## Grid Search for tuning the hyperparameters of the XGBoost

In [137]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [300, 400, 500],
    "learning_rate": [0.03, 0.05, 0.07],
    "max_depth": [5, 6, 7]
}

search = GridSearchCV(XGBRegressor(), param_grid, cv=5, scoring="neg_mean_absolute_error", verbose=1)
search.fit(X_train, y_train)

print("Best params:", search.best_params_)
print("Best MAE:", -search.best_score_)

Fitting 5 folds for each of 27 candidates, totalling 135 fits
Best params: {'learning_rate': 0.07, 'max_depth': 5, 'n_estimators': 300}
Best MAE: 198048.55256737524


In [140]:
best_xgb = XGBRegressor(learning_rate=0.07, max_depth=5, n_estimators=300)
best_xgb.fit(X_train, y_train)

y_pred_best = best_xgb.predict(X_test)
print("Tuned XGBoost MAE:", mean_absolute_error(y_test, y_pred_best))

Tuned XGBoost MAE: 166274.975440492


## Bargain Finder for finding the listings where the listed price is a lot lower than our predicted price

In [141]:
df['predicted_price'] = best_xgb.predict(X)
df['discount'] = df['predicted_price'] - df['price']

bargains = df[['url', 'brand_series', 'year', 'mileage', 'price', 'predicted_price', 'discount', 'heavy_damage', 'paint_changed']]
bargains = bargains.sort_values('discount', ascending=False)
bargains.head(20)

,url,brand_series,year,mileage,price,predicted_price,discount,heavy_damage,paint_changed
331,https://www.arabam.com/ilan/galeriden-satilik-...,Mercedes - Benz_S,2003,342000.0,1350000.0,2.855614e+06,1.505614e+06,Belirtilmemiş,Tamamı orjinal
704,https://www.arabam.com/ilan/galeriden-satilik-...,Mercedes - Benz_E,2023,53500.0,4850000.0,5.958112e+06,1.108112e+06,Belirtilmemiş,Belirtilmemiş
220,https://www.arabam.com/ilan/galeriden-satilik-...,Mercedes - Benz_E,1986,222520.0,175500.0,1.275007e+06,1.099507e+06,Evet,Belirtilmemiş
588,https://www.arabam.com/ilan/galeriden-satilik-...,Mercedes - Benz_E,2023,53000.0,5250000.0,5.958112e+06,7.081115e+05,Belirtilmemiş,Tamamı orjinal
1587,https://www.arabam.com/ilan/sahibinden-satilik...,Hyundai_Accent Blue,2017,120000.0,365000.0,9.659611e+05,6.009611e+05,Hayır,Tamamı boyalı
1875,https://www.arabam.com/ilan/galeriden-satilik-...,Hyundai_Accent Blue,2016,182000.0,364750.0,9.370354e+05,5.722854e+05,Belirtilmemiş,Tamamı boyalı
49,https://www.arabam.com/ilan/galeriden-satilik-...,Volvo_S90,2021,110000.0,3360000.0,3.898547e+06,5.385472e+05,Belirtilmemiş,4 boyalı
1580,https://www.arabam.com/ilan/galeriden-satilik-...,Hyundai_Accent Blue,2016,120000.0,424500.0,8.716312e+05,4.471312e+05,Belirtilmemiş,Tamamı boyalı
559,https://www.arabam.com/ilan/galeriden-satilik-...,Mercedes - Benz_CLA,2023,34000.0,2950000.0,3.396735e+06,4.467348e+05,Belirtilmemiş,"1 değişen, 1 boyalı"
583,https://www.arabam.com/ilan/galeriden-satilik-...,Fiat_Egea,2022,92000.0,870000.0,1.303407e+06,4.334066e+05,Belirtilmemiş,Tamamı orjinal
